In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
import seaborn as sns

import pickle

sns.set_style('darkgrid')

from catboost import CatBoostClassifier

from sklearn.preprocessing import PolynomialFeatures

from statannotations.Annotator import Annotator

from scipy.stats import shapiro, chi2_contingency

import itertools

In [3]:
df_main = pd.read_excel('../data/df_main.xlsx')
display(df_main.head())
df_main.info()

,test,no_fragmentation,splashing,splashing_spectrum,breaking_up,rebound,one_drop,voltage,long_impulse_duration,long_impulse_dur_binary,...,sedimentation_velocity,sedimentation_Re,sedimentation_Stk,sign_sedimentation_Re,sign_sedimentation_Stk,sign_particle_droplet_diameter_ratio,droplet_density,free_fall_velocity,drag_velocity,relative_roughness
0,3,0,1,2,0,0,1,105.0,10,low,...,0.000018,0.000092,8.252778e-08,0.000092,8.252778e-08,0.013301,836.828587,3.961141,3.682513,0.000032
1,4,0,1,2,0,0,1,105.0,10,low,...,0.000018,0.000092,8.252778e-08,0.000092,8.252778e-08,0.013301,835.298244,3.961141,3.682027,0.000032
2,5,0,1,2,0,0,1,105.0,10,low,...,0.000018,0.000092,8.252778e-08,0.000092,8.252778e-08,0.013301,835.298244,3.961141,3.682027,0.000032
3,7,0,1,2,0,0,0,105.0,10,low,...,0.000018,0.000092,8.582889e-08,0.000092,8.582889e-08,0.013833,836.869874,3.961141,3.668435,0.000013
4,8,0,1,2,0,0,0,105.0,10,low,...,0.000018,0.000092,8.582889e-08,0.000092,8.582889e-08,0.013833,836.795029,3.961141,3.668410,0.000013


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 42 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   test                                  372 non-null    int64  
 1   no_fragmentation                      372 non-null    int64  
 2   splashing                             372 non-null    int64  
 3   splashing_spectrum                    372 non-null    int64  
 4   breaking_up                           372 non-null    int64  
 5   rebound                               372 non-null    int64  
 6   one_drop                              372 non-null    int64  
 7   voltage                               372 non-null    float64
 8   long_impulse_duration                 372 non-null    int64  
 9   long_impulse_dur_binary               372 non-null    object 
 10  wettability                           372 non-null    object 
 11  roughness          

In [5]:
df_main['Re']

0      1387.532037
1      1387.348917
2      1387.348917
3      1329.064815
4      1329.055807
          ...     
367     312.785172
368     308.961356
369     306.095674
370     313.740707
371     319.475814
Name: Re, Length: 372, dtype: float64

In [11]:
df_main.columns

Index(['test', 'no_fragmentation', 'splashing', 'splashing_spectrum',
       'breaking_up', 'rebound', 'one_drop', 'voltage',
       'long_impulse_duration', 'long_impulse_dur_binary', 'wettability',
       'roughness', 'liquid_density', 'surface_tension', 'viscosity',
       'particle_mean_diameter', 'particle_density', 'volume_fraction',
       'droplet_diameter', 'height', 'inclination', 'roughness_binary',
       'particle_liquid_density_ratio', 'volume_fraction_binary',
       'particle_diameter_cat', 'particle_droplet_diameter_ratio', 'velocity',
       'Re', 'We', 'K', 'no_mixing_time', 'init_volume_fraction',
       'sedimentation_velocity', 'sedimentation_Re', 'sedimentation_Stk',
       'sign_sedimentation_Re', 'sign_sedimentation_Stk',
       'sign_particle_droplet_diameter_ratio', 'droplet_density',
       'free_fall_velocity', 'drag_velocity', 'relative_roughness'],
      dtype='object')

In [14]:
df_main.groupby('volume_fraction_binary')[['We', 'Re', 'K']].median()

,We,Re,K
volume_fraction_binary,,,
0,764.673342,654.762119,146.804385
1,614.418761,617.384895,133.485239


In [26]:
df_main.groupby('volume_fraction_binary')[['droplet_diameter', 'liquid_density', 'viscosity', 'surface_tension']].median()

,droplet_diameter,liquid_density,viscosity,surface_tension
volume_fraction_binary,,,,
0,0.00324,1180.0,0.0231,0.0679
1,0.00322,1180.0,0.0231,0.0679


In [30]:
df_main.groupby('volume_fraction_binary')[['free_fall_velocity', 'drag_velocity', 'velocity']].median()

,free_fall_velocity,drag_velocity,velocity
volume_fraction_binary,,,
0,3.961141,3.756316,3.756316
1,3.961141,3.744056,3.546038


In [33]:
df_main.groupby('volume_fraction_binary')['inclination'].mean()

volume_fraction_binary
0    0.000000
1    0.232818
Name: inclination, dtype: float64

Таким образом, косвенное влияние phi_drop на K-parameter вызвано тем, что с наклонными подложками больше экспериментов проводилось с жидкостями большей концентрации, чем с горизонтальными.